# Compare approaches — LV / IP segmentation

Aggregate the per-case metrics from any set of runs and compare them (Dice violins,
MD/FA GT-vs-prediction, insertion-point Hausdorff) + save a tidy **CSV** for your own plots.

**How to use:** each run of `process_folders` (crop pipeline, full-scale, YOLO IP variants,
different dataset IDs / encoders) writes a per-case `.xlsx`. Just point `RUNS` at those files
with a human label each. To compare a new model, add one line — nothing else changes.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = '/home/sastocke/nnUNet'

# ============================================================================
#  Add / edit / remove lines to compare whatever runs you like.
#  label -> the per-case metrics .xlsx that process_folders wrote for that run.
#  (a glob is allowed; the newest match is used, so you can leave wildcards in.)
# ============================================================================
RUNS = {
    'crop D100 | YOLO-mc IP': f'{BASE}/inferenceTest_SmartHealthTestHannum_D100/SmartHealthTestHannum_D100_yolo_mc.xlsx',
    'crop D100 | YOLO-1c IP': f'{BASE}/inferenceTest_SmartHealthTestHannum_D100/SmartHealthTestHannum_D100_yolo_1c.xlsx',
    # 'crop D200 | YOLO-mc IP': f'{BASE}/inferenceTest_SmartHealthTestHannum_D200/SmartHealthTestHannum_D200_yolo_mc.xlsx',  # <- swap the LV dataset ID, rerun, add a line
    # 'crop D2xx (ResEnc-L)' : f'{BASE}/inferenceTest_.../....xlsx',
}

OUT_DIR = f'{BASE}/comparisons'; os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = f'{OUT_DIR}/comparison_percase.csv'      # tidy per-case rows (for your own plotting)
OUT_SUMMARY = f'{OUT_DIR}/comparison_summary.csv'  # mean/median/std per approach

# the metric columns process_folders writes (kept if present)
DICE   = 'Dice Score Original Label 1'
HD2, HD3 = 'Hausdorff Distance Label 2', 'Hausdorff Distance Label 3'
MD_GT, MD_PR = 'GT_Median_MD', 'Pred_median_MD'
FA_GT, FA_PR = 'GT_Median_FA', 'Pred_median_FA'
HD_MISS = 1250.0   # process_folders failsafe for a MISSED insertion point (1000 x 1.25)


## 1. Load all runs → one tidy DataFrame + CSV

In [ ]:
def _resolve(path):
    if os.path.exists(path):
        return path
    hits = sorted(glob.glob(path), key=os.path.getmtime)
    return hits[-1] if hits else None

frames = []
for label, path in RUNS.items():
    p = _resolve(path)
    if p is None:
        print(f'!! no file for "{label}": {path}'); continue
    df = pd.read_excel(p)
    df.insert(0, 'approach', label)
    frames.append(df)
    print(f'loaded "{label}": {len(df)} cases  <- {os.path.basename(p)}')

alldf = pd.concat(frames, ignore_index=True)
order = list(RUNS.keys())                     # keep the config order on the x-axis
alldf['approach'] = pd.Categorical(alldf['approach'], categories=order, ordered=True)
alldf.to_csv(OUT_CSV, index=False)
print(f'\nwrote {OUT_CSV}  ({len(alldf)} rows, {alldf.approach.nunique()} approaches)')
alldf.head()


## 2. Dice violins by approach (the headline figure)

In [ ]:
def violin(ax, groups, labels, ylabel, title, points=True):
    vals = [np.asarray(g, float) for g in groups]
    vals = [v[np.isfinite(v)] for v in vals]
    parts = ax.violinplot(vals, showmedians=True, showextrema=False, widths=0.8)
    for b in parts['bodies']:
        b.set_alpha(0.45)
    if 'cmedians' in parts:
        parts['cmedians'].set_color('k'); parts['cmedians'].set_linewidth(2)
    if points:
        for i, v in enumerate(vals):
            ax.scatter(np.random.normal(i + 1, 0.05, len(v)), v, s=12, alpha=0.5,
                       color='k', zorder=3, linewidths=0)
    for i, v in enumerate(vals):                      # median label
        if len(v): ax.text(i + 1, np.median(v), f'{np.median(v):.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels, rotation=20, ha='right')
    ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis='y', alpha=0.3)

fig, ax = plt.subplots(figsize=(1.8 * len(order) + 3, 5))
violin(ax, [alldf[alldf.approach == a][DICE].values for a in order], order,
       'Dice (LV, label 1)', 'LV Dice — automatic vs GT, by approach')
ax.set_ylim(0, 1)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/dice_violin.png', dpi=200); plt.show()


## 3. Insertion-point accuracy (Hausdorff, px→mm) by approach
`1250` is the failsafe for a *missed* point — reported separately as a miss rate so it
doesn't swamp the violin of the detected points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(1.8 * len(order) + 4, 5))
for ax, hd, name in zip(axes, [HD2, HD3], ['anterior (label 2)', 'inferior (label 3)']):
    if hd not in alldf.columns:
        ax.set_visible(False); continue
    groups, miss = [], []
    for a in order:
        v = alldf[alldf.approach == a][hd].values
        miss.append(f'{a}: {(v >= HD_MISS).mean() * 100:.0f}% missed')
        groups.append(v[v < HD_MISS])            # detected only
    violin(ax, groups, order, 'Hausdorff [mm]', f'IP {name} (detected only)')
    print('\n'.join(miss)); print()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/ip_hausdorff_violin.png', dpi=200); plt.show()


## 4. MD / FA / DWI — GT vs Prediction (median_boxplot.ipynb style)

Split violin per approach: **left half = GT (grey)**, **right half = Pred (coloured)**, `inner='quartile'`, with the paired per-case points and connecting lines so you can see how each case's prediction tracks its ground truth. One figure per detected metric.

In [ ]:
import seaborn as sns, re
from matplotlib.lines import Line2D

# nice axis labels (+ optional y-limits) per metric; unknown metrics fall back to the raw name.
METRIC_LABELS = {
    'MD':  ('Mean Diffusivity [$\\mu m^2 / ms$]', None),
    'FA':  ('Fractional Anisotropy', None),
    'DWI': ('DWI signal [a.u.]', None),
}
CASE = 'Case ID'

def gt_pred_split_violin(metric, gt_col, pred_col):
    ylabel, ylim = METRIC_LABELS.get(metric, (metric, None))
    m = alldf[[CASE, 'approach', gt_col, pred_col]].copy()
    melted = m.melt(id_vars=[CASE, 'approach'], value_vars=[gt_col, pred_col],
                    var_name='Type', value_name='Median')
    melted['Type'] = melted['Type'].replace({gt_col: 'GT', pred_col: 'Pred'})

    approaches = [a for a in order if a in set(melted['approach'].astype(str))]
    pred_colors = {a: plt.cm.tab10(i % 10) for i, a in enumerate(approaches)}
    palette = {'GT': 'grey', 'Pred': 'black'}

    plt.figure(figsize=(2.4 * len(approaches) + 2, 4), dpi=300)
    plt.title(f'{metric} — GT vs Prediction', fontsize=18)
    sns.violinplot(x='approach', y='Median', hue='Type', data=melted, split=True,
                   inner='quartile', palette=palette, cut=0, width=0.5, order=approaches)

    # grey GT half (left), dataset-specific colour on the Pred half (right)
    for idx, a in enumerate(approaches):
        for patch in plt.gca().collections[2 * idx:2 * idx + 2]:
            patch.set_facecolor('grey' if patch.get_paths()[0].vertices[:, 0].min() < idx else pred_colors[a])
            patch.set_edgecolor('black'); patch.set_alpha(0.6)

    # paired per-case points + connecting lines (GT at idx-0.1, Pred at idx+0.1)
    for idx, a in enumerate(approaches):
        d = melted[melted['approach'] == a]
        gt = d[d['Type'] == 'GT'].sort_values(CASE)['Median'].values
        pr = d[d['Type'] == 'Pred'].sort_values(CASE)['Median'].values
        plt.scatter(np.full(len(gt), idx - 0.1) + np.random.uniform(-0.02, 0.02, len(gt)), gt,
                    color='black', s=10, alpha=0.7)
        plt.scatter(np.full(len(pr), idx + 0.1) + np.random.uniform(-0.02, 0.02, len(pr)), pr,
                    color='black', s=10, alpha=0.7)
        for g, prd in zip(gt, pr):
            plt.plot([idx - 0.1, idx + 0.1], [g, prd], color='black', alpha=0.3)

    plt.xticks(fontsize=13, rotation=15, ha='right'); plt.yticks(fontsize=12)
    plt.ylabel(ylabel, fontsize=14); plt.xlabel('')
    if ylim: plt.ylim(*ylim)
    plt.legend(handles=[Line2D([0], [0], color='grey', lw=6, label='GT')], fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/gtpred_{metric}.png', dpi=300); plt.show()

# auto-detect every GT_Median_X / Pred_median_X pair present -> one figure each
pairs = []
for c in alldf.columns:
    mm = re.match(r'GT_Median_(\w+)$', c)
    if mm and f'Pred_median_{mm.group(1)}' in alldf.columns:
        pairs.append((mm.group(1), c, f'Pred_median_{mm.group(1)}'))
print('GT-vs-Pred metrics found:', [p[0] for p in pairs])
for metric, gt_col, pred_col in pairs:
    gt_pred_split_violin(metric, gt_col, pred_col)


In [ ]:
from scipy.stats import ttest_ind
# Welch t-test GT vs Pred per approach & metric (as in median_boxplot.ipynb).
for metric, gt_col, pred_col in pairs:
    print(f'[{metric}]  GT vs Pred (Welch t-test)')
    for a in order:
        sub = alldf[alldf['approach'] == a]
        gt = sub[gt_col].astype(float).dropna().values
        pr = sub[pred_col].astype(float).dropna().values
        if len(gt) and len(pr):
            t, pv = ttest_ind(gt, pr, equal_var=False)
            print(f'   {a}: t={t:.4f}  p={pv:.4f}')
    print()


## 5. Summary table → CSV

In [ ]:
metrics = [c for c in [DICE, 'F1 Label 1', 'Precision', 'Recall', HD2, HD3,
                       MD_GT, MD_PR, FA_GT, FA_PR, 'Avg. HD Epi', 'Avg. HD Endo'] if c in alldf.columns]
summary = alldf.groupby('approach', observed=True)[metrics].agg(['mean', 'median', 'std'])
summary.to_csv(OUT_SUMMARY)
print('wrote', OUT_SUMMARY)
summary.round(3)
